# toy_sc_m — Julia single-column model (notebook)

Kernel: select the Julia kernel (IJulia).

Usage:
- Run cells top-to-bottom. The final `run_experiment` cell writes `toy_sc_m_results.csv` to `../ML/`.
- If IJulia is not installed: in Julia REPL run `import Pkg; Pkg.add("IJulia")`.


In [ ]:
# Imports and physical constants
using Printf, LinearAlgebra, Random, DelimitedFiles

const g = 9.80665              # m/s^2
const cp = 1004.0              # J/(kg K)
const Lv = 2.5e6               # J/kg
const sigma = 5.670374419e-8   # W/m2/K4
const rho_air = 1.225          # kg/m3 (near surface)
const CH = 1.2e-3
const CE = 1.0e-3

In [ ]:
# Grid type and helper to create a stretched grid
struct Grid
    z::Vector{Float64}
    dz::Vector{Float64}
    nz::Int
end

function create_stretched_grid(z_top::Float64, nz::Int; dz_min=5.0)
    η = range(0.0, 1.0, length=nz)
    stretch = (exp.(3*η) .- 1.0) ./ (exp(3.0)-1.0)
    z = z_top .* stretch
    dz = similar(z)
    dz[1] = max(dz_min, z[2]-z[1])
    for i in 2:nz-1
        dz[i] = z[i+1]-z[i-1]
    end
    dz[end] = z[end]-z[end-1]
    return Grid(z, dz, nz)
end

In [ ]:
# State container and initializer
mutable struct State
    T::Vector{Float64}
    q::Vector{Float64}
    Ts::Float64
    ice_h::Float64
end

function initialize_state(grid::Grid)
    nz = grid.nz
    T = zeros(nz); q = zeros(nz)
    for i in 1:nz
        T[i] = 250.0 + 0.015 * grid.z[i]
        q[i] = 3e-4 * exp(-grid.z[i]/2000)
    end
    Ts = 258.0; ice_h = 1.0
    return State(T, q, Ts, ice_h)
end

In [ ]:
# Surface and radiation helpers
function albedo(ice_h::Float64, snow_mass::Float64=0.0, meltpond_frac::Float64=0.0)
    alb_ice = 0.6; alb_snow = 0.85; alb_pond = 0.2
    if snow_mass > 0.0
        return alb_snow*(1.0-meltpond_frac) + alb_pond*meltpond_frac
    else
        frac = clamp(ice_h/0.3, 0.0, 1.0)
        return alb_pond*(1-frac) + alb_ice*frac
    end
end

function radiation_simple(Ts::Float64, alb::Float64, sw_down::Float64, lw_down::Float64)
    sw_absorbed = sw_down * (1.0 - alb)
    lw_up = sigma * Ts^4
    lw_net = lw_down - lw_up
    rnet = sw_absorbed + lw_net
    return rnet, sw_absorbed, lw_net
end

function surface_fluxes(Ts::Float64, T1::Float64, q1::Float64, U::Float64; rho=rho_air)
    H = rho*cp*CH*U*(Ts - T1)
    LE = rho*Lv*CE*U*(0.0 - q1)
    return H, LE
end

In [ ]:
# Saturation and K-profile
function q_sat(T::Float64, p::Float64=900.0)
    es = 6.112 * exp((17.67*(T-273.15))/(T-29.65)) * 100.0
    qsat = 0.622 * es / (p*100.0 - 0.378*es)
    return qsat
end

function compute_K_profile(grid::Grid, state::State; K0=1.0, hbl=200.0, minK=1e-5)
    nz = grid.nz; K = zeros(nz)
    for i in 1:nz
        z = grid.z[i]; K[i] = K0 * exp(-z/hbl)
    end
    if state.Ts < state.T[1]
        inv_strength = state.T[1] - state.Ts
        factor = max(0.05, 1.0 - 0.2*inv_strength)
        K .= K .* factor
    end
    K .= max.(K, minK); return K
end

In [ ]:
# Vertical diffusion tendency and step function
function vertical_diffusion_tend(grid::Grid, var::Vector{Float64}, K::Vector{Float64})
    nz = grid.nz; tend = zeros(nz); F = zeros(nz+1)
    for k in 2:nz
        Km = 0.5*(K[k] + K[k-1])
        dvar = (var[k] - var[k-1]) / (grid.z[k] - grid.z[k-1])
        F[k] = -Km * dvar
    end
    F[1] = 0.0; F[nz+1] = 0.0
    for i in 1:nz
        dz = grid.dz[i]; tend[i] = -(F[i+1] - F[i]) / dz
    end
    return tend
end

function step!(grid::Grid, state::State, dt::Float64, forcings)
    alb = albedo(state.ice_h)
    rnet, sw_abs, lw_net = radiation_simple(state.Ts, alb, forcings[:sw_down], forcings[:lw_down])
    H, LE = surface_fluxes(state.Ts, state.T[1], state.q[1], forcings[:U])
    K = compute_K_profile(grid, state; K0=0.5, hbl=100.0, minK=1e-6)
    tendT = vertical_diffusion_tend(grid, state.T, K)
    tendq = vertical_diffusion_tend(grid, state.q, K)
    for i in 1:grid.nz
        state.T[i] += dt*(tendT[i] + (forcings[:adv_T] === nothing ? 0.0 : forcings[:adv_T][i]))
        state.q[i] += dt*(tendq[i] + (forcings[:adv_q] === nothing ? 0.0 : forcings[:adv_q][i]))
    end
    k_ice = 2.1; T_deep = 258.0
    G = k_ice*(state.Ts - T_deep) / max(state.ice_h, 0.01)
    dTs = (rnet - (H + LE) - G) * dt / (2100.0 * state.ice_h)
    state.Ts += dTs
    dz1 = grid.dz[1]
    state.T[1] += dt * (H / (rho_air * cp * dz1))
    state.q[1] += dt * (LE / (rho_air * Lv * dz1))
    for i in 1:grid.nz
        qsat = q_sat(state.T[i])
        if state.q[i] > qsat
            dq = state.q[i] - qsat
            dTlat = - (Lv * dq) / cp
            state.T[i] += dTlat; state.q[i] = qsat
        end
    end
    if state.Ts > 273.15
        melt_rate = 1e-6*(state.Ts - 273.15)
        state.ice_h = max(0.0, state.ice_h - melt_rate*dt)
    end
    return nothing
end

In [ ]:
# Driver: run_experiment and save CSV
function run_experiment(; ztop=4000.0, nz=40, dt=60.0, tmax=24*3600.0)
    grid = create_stretched_grid(ztop, nz)
    state = initialize_state(grid)
    sw_max = 200.0; lw_down = 250.0; U = 5.0
    times = 0.0:dt:tmax; nsteps = length(times)
    Ts_hist = zeros(nsteps); T1_hist = zeros(nsteps); ice_h_hist = zeros(nsteps)
    for (it, t) in enumerate(times)
        dayfrac = (t / 86400.0) % 1.0
        sw_down = max(0.0, sw_max * sin(2π*dayfrac))
        forcings = Dict(:sw_down=>sw_down, :lw_down=>lw_down, :U=>U, :adv_T=>nothing, :adv_q=>nothing)
        step!(grid, state, dt, forcings)
        Ts_hist[it] = state.Ts; T1_hist[it] = state.T[1]; ice_h_hist[it] = state.ice_h
        if it % 240 == 1
            @printf("t=%.1f h  Ts=%.2f K  T1=%.2f K  ice_h=%.3f m  sw=%.1f W/m2\n", t/3600, state.Ts, state.T[1], state.ice_h, sw_down)
        end
    end
    return times, Ts_hist, T1_hist, ice_h_hist
end

# Run and save results (uncomment to run in notebook)
# times, Ts_hist, T1_hist, ice_h_hist = run_experiment(ztop=4000.0, nz=40, dt=60.0, tmax=24*3600.0)
# out = hcat(collect(times), Ts_hist, T1_hist, ice_h_hist)
# csv_path = joinpath(@__DIR__, "..", "ML", "toy_sc_m_results.csv")
# mkpath(dirname(csv_path)); writedlm(csv_path, out, ','); println("Saved -> ", csv_path)

### How to run
- In a Julia kernel: evaluate the last code cell but first remove the leading `#` on the three lines that call `run_experiment` and save CSV.
- The saved CSV is `../ML/toy_sc_m_results.csv` relative to this notebook. Python notebook cells can then read and plot it.